In [0]:
import re
import pyspark.sql.functions as F
from pyspark.sql.functions import col, get_json_object, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.transaction_events")

def StandardizeNames(df):
    l = df.columns
    cols = []
    for c in l:
        temp = re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower()
        for _ in range(5):
            temp = re.sub(r'\b([a-z])_([a-z])\b', r'\1\2', temp)
            temp = re.sub(r'(?<=[a-z])_([a-z])(?=_|$)', r'\1', temp)
        temp = re.sub(r'_+','_', temp)
        temp = temp.lstrip('_')
        cols.append(temp)
    return df.toDF(*cols)
df = StandardizeNames(df)
df.dtypes

In [0]:
# Extracting
df = df.withColumn('txn_entry_mode', get_json_object(col('pos_entry_details'), '$.mode'))
df = df.withColumn('card_type',      get_json_object(col('pos_entry_details'), '$.type'))
df.dtypes


In [0]:
# Deleting duplicated data
df.dropDuplicates(['transaction_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['transaction_id','file_path','ingest_datetime'])
df = df.drop("kafka_topics", "pos_entry_details")

In [0]:
# Altering values diferent than 'APPROVED'
df = df.withColumn('txn_status', F.when((col('txn_status').startswith('A')) | (col('txn_status').startswith('a')), 'APPROVED').otherwise(col('txn_status')))

# Altering values that are not 'CREDIT' or 'DEBIT'
df = df.withColumn('card_type', regexp_replace(col('card_type'), r'\\u00.*9', ''))
df = df.withColumn('card_type', F.when(col('card_type').startswith('C'), 'CREDIT').otherwise(col('card_type')))
df = df.withColumn('card_type', F.when(col('card_type').startswith('D'), 'DEBIT').otherwise(col('card_type')))


In [0]:

target = "fraud_detection_project.silver_layer.transaction_events"

if spark.catalog.tableExists(target):
    dt = DeltaTable.forName(spark, target)

    dt.alias("t").merge(
        df.alias("s"),
        "t.transaction_id = s.transaction_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Merge concluded.")
else:
    df.write.format("delta") \
      .option("mergeSchema", "true") \
      .saveAsTable(target)
    print("Tabel created.")

In [0]:
%skip
df.createOrReplaceTempView('df1')

In [0]:
%skip
SELECT card_type, txn_status FROM df1;